In [ ]:
!pip install -U ultralytics roboflow pyyaml pandas

In [ ]:
# =========================
# Project Config
# =========================

ROBOFLOW_WORKSPACE = "2026-oss"
ROBOFLOW_PROJECT = "ai-picture-book-object-detection"
ROBOFLOW_VERSION = 15

MODEL_NAME = "yolo11n.pt"
RUN_NAME = f"picture_book_yolo11n_v{ROBOFLOW_VERSION}"

IMG_SIZE = 640
EPOCHS = 150
BATCH_SIZE = 64

PREDICT_CONF_THRESHOLD = 0.15
PREDICT_IOU_THRESHOLD = 0.80
MAX_DETECTIONS = 50

SEED = 42

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    !nvidia-smi
else:
    raise RuntimeError("GPU가 연결되어 있지 않습니다. Colab 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.")

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

    if not ROBOFLOW_API_KEY:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW")
except Exception:
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass("Roboflow Private API Key를 입력하세요: ")

ROBOFLOW_API_KEY = ROBOFLOW_API_KEY.strip()

print("Roboflow API Key loaded:", len(ROBOFLOW_API_KEY) > 0)
print("API Key length:", len(ROBOFLOW_API_KEY))

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

print("Dataset version:", ROBOFLOW_VERSION)
print("Dataset downloaded to:", dataset.location)

In [ ]:
import os
import yaml

yaml_path = os.path.join(dataset.location, "data.yaml")

with open(yaml_path, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

data_yaml["train"] = os.path.join(dataset.location, "train", "images")
data_yaml["val"] = os.path.join(dataset.location, "valid", "images")
data_yaml["test"] = os.path.join(dataset.location, "test", "images")

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print("Updated data.yaml:")
with open(yaml_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
EXPECTED_CLASSES = [
    "book_flower",
    "book_flowerpot",
    "book_monkey",
    "book_stone",
    "braille",
    "tactile_flower",
    "tactile_flowerpot",
    "tactile_monkey",
    "tactile_stone",
    "text",
]

with open(yaml_path, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

names = data_yaml.get("names")

if isinstance(names, dict):
    actual_classes = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
else:
    actual_classes = names

print("Actual classes:")
for idx, name in enumerate(actual_classes):
    print(f"{idx}: {name}")

print("
Expected class count:", len(EXPECTED_CLASSES))
print("Actual class count:", len(actual_classes))

missing_classes = sorted(set(EXPECTED_CLASSES) - set(actual_classes))
extra_classes = sorted(set(actual_classes) - set(EXPECTED_CLASSES))

print("
Missing classes:", missing_classes)
print("Extra classes:", extra_classes)

assert len(actual_classes) == len(EXPECTED_CLASSES), "클래스 개수가 예상과 다릅니다."
assert set(actual_classes) == set(EXPECTED_CLASSES), "클래스 목록이 예상과 다릅니다."

print("
클래스 검증 완료")

In [ ]:
import glob

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(dataset.location, split, "images")
    label_dir = os.path.join(dataset.location, split, "labels")

    image_count = len(glob.glob(os.path.join(image_dir, "*")))
    label_count = len(glob.glob(os.path.join(label_dir, "*")))

    print(f"{split}: images={image_count}, labels={label_count}")

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=yaml_path,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    name=RUN_NAME,
    project="runs/detect",
    device=0,
    seed=SEED,
    deterministic=True,
    patience=30,
    cos_lr=True,
    cache=True,
    plots=False,
    exist_ok=True,
)

In [ ]:
from pathlib import Path

run_dir = Path("runs/detect") / RUN_NAME
best_model_path = run_dir / "weights" / "best.pt"
last_model_path = run_dir / "weights" / "last.pt"

assert best_model_path.exists(), f"best.pt 파일을 찾지 못했습니다: {best_model_path}"

best_model_path = str(best_model_path)

print("Run directory:", run_dir)
print("Best model path:", best_model_path)
print("Last model path:", last_model_path)

In [ ]:
best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=yaml_path,
    split="test",
    imgsz=IMG_SIZE,
    plots=False,
)

In [ ]:
summary = {
    "model": MODEL_NAME,
    "dataset_version": ROBOFLOW_VERSION,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP@0.5": float(metrics.box.map50),
    "mAP@0.5:0.95": float(metrics.box.map),
    "best_model_path": best_model_path,
}

summary

In [ ]:
import pandas as pd

class_result_rows = []

ap50_values = getattr(metrics.box, "ap50", None)
ap_values = getattr(metrics.box, "ap", None)

for idx, class_name in enumerate(actual_classes):
    row = {
        "class_id": idx,
        "class_name": class_name,
        "mAP@0.5": float(ap50_values[idx]) if ap50_values is not None and idx < len(ap50_values) else None,
        "mAP@0.5:0.95": float(ap_values[idx]) if ap_values is not None and idx < len(ap_values) else None,
    }
    class_result_rows.append(row)

class_results_df = pd.DataFrame(class_result_rows)
class_results_df

In [ ]:
import json

os.makedirs("results", exist_ok=True)

summary_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_summary.json"
class_result_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_class_results.csv"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

class_results_df.to_csv(class_result_path, index=False, encoding="utf-8-sig")

print("Saved summary:", summary_path)
print("Saved class results:", class_result_path)

In [ ]:
test_image_path = os.path.join(dataset.location, "test", "images")

predict_results = best_model.predict(
    source=test_image_path,
    conf=PREDICT_CONF_THRESHOLD,
    iou=PREDICT_IOU_THRESHOLD,
    max_det=MAX_DETECTIONS,
    save=True,
    name=f"{RUN_NAME}_test_predictions",
    project="runs/detect",
    exist_ok=True,
)

print("Prediction settings")
print("conf:", PREDICT_CONF_THRESHOLD)
print("iou:", PREDICT_IOU_THRESHOLD)
print("max_det:", MAX_DETECTIONS)

In [ ]:
import shutil

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

final_model_path = f"models/yolo11n_v{ROBOFLOW_VERSION}_best.pt"
shutil.copy(best_model_path, final_model_path)

print("Saved final model:", final_model_path)

args_src_path = Path(best_model_path).parents[1] / "args.yaml"
args_dst_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_args.yaml"

if args_src_path.exists():
    shutil.copy(args_src_path, args_dst_path)
    print("Saved args.yaml:", args_dst_path)
else:
    print("args.yaml 파일을 찾지 못했습니다:", args_src_path)

In [ ]:
model_info = {
    "model_name": MODEL_NAME,
    "run_name": RUN_NAME,
    "dataset_version": ROBOFLOW_VERSION,
    "input": {
        "type": "image_or_webcam_frame",
        "image_size": IMG_SIZE,
    },
    "output": {
        "class_name": "detected object class",
        "confidence": "detection confidence score",
        "bbox": "[x1, y1, x2, y2]",
    },
    "classes": actual_classes,
    "best_model_path": best_model_path,
    "final_model_path": final_model_path,
    "metrics": summary,
    "prediction_settings": {
        "conf": PREDICT_CONF_THRESHOLD,
        "iou": PREDICT_IOU_THRESHOLD,
        "max_det": MAX_DETECTIONS,
    },
    "integration_notes": {
        "finger_recognition": (
            "손가락 인식은 YOLO 학습 노트북이 아니라 Backend/MediaPipe Hands 단계에서 처리한다. "
            "YOLO 모델은 손끝 좌표와 매칭할 객체 bbox를 제공하는 역할이다."
        ),
        "close_object_detection": (
            "가까운 객체가 동시에 탐지되는지 확인하기 위해 prediction 단계에서 conf를 낮추고 "
            "NMS IoU threshold를 높게 설정했다."
        ),
    },
}

model_info_path = f"results/yolo11n_v{ROBOFLOW_VERSION}_model_info.json"

with open(model_info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print("Saved model info:", model_info_path)

In [ ]:
from google.colab import files

zip_name = f"yolo11n_v{ROBOFLOW_VERSION}_results.zip"

!zip -r {zip_name} models results

files.download(zip_name)